In [ ]:
from jupytergis import GISDocument
from IPython.display import display

latitude = 47.52
longitude = -2.95

doc = GISDocument(latitude=latitude, longitude=longitude, zoom=12)
doc

In [ ]:
doc.add_raster_layer('https://gibs.earthdata.nasa.gov/wmts/epsg3857/best/ASTER_GDEM_Greyscale_Shaded_Relief/default/GoogleMapsCompatible_Level12/{z}/{y}/{x}.jpg')

In [ ]:
from rio_tiler.models import ImageData
import numpy as np

latest_stack = None


def ndwi_process(stack):
    global latest_stack
    latest_stack = stack
    
    h = stack.sizes.get("y", 256)
    w = stack.sizes.get("x", 256)

    if "time" in stack.dims and stack.sizes["time"] == 0:
        return ImageData(np.ma.masked_all((1, h, w), dtype=np.uint8))

    data = stack
    if "time" in data.dims:
        # pick one scene OR use median over time; choose one
        data = data.isel(time=0)
        # data = data.median(dim="time", skipna=True)

    if "band" not in data.dims or data.sizes["band"] < 2:
        return ImageData(np.ma.masked_all((1, h, w), dtype=np.uint8))

    try:
        green = data.sel(band="green").data.astype(np.float32)
        nir = data.sel(band="nir").data.astype(np.float32)
    except Exception:
        green = data.isel(band=0).data.astype(np.float32)
        nir = data.isel(band=1).data.astype(np.float32)

    if hasattr(green, "compute"):
        green = green.compute()
    if hasattr(nir, "compute"):
        nir = nir.compute()

    ndwi = (green - nir) / (green + nir + 1e-6)
    ndwi = np.asarray(ndwi)  # drop masked-array dimensional surprises
    ndwi = np.nan_to_num(ndwi, nan=-1.0, posinf=1.0, neginf=-1.0)

    # ensure 2D before adding band axis
    if ndwi.ndim == 3:
        ndwi = ndwi[0]
    if ndwi.ndim != 2:
        return ImageData(np.ma.masked_all((1, h, w), dtype=np.uint8))

    ndwi_01 = np.clip((ndwi + 1.0) / 2.0, 0.0, 1.0)
    pixels = (ndwi_01 * 255).astype(np.uint8)  # 2D
    return ImageData(pixels[np.newaxis, :, :])  # 3D: 1, y, x


await doc.add_stac_array_layer(
    stac_url="https://earth-search.aws.element84.com/v1",
    array_to_image=ndwi_process,
    collection_id="sentinel-2-l2a",
    assets=["green", "nir"],
    datetime="2024-06-01/2024-09-30",
    query={"eo:cloud_cover": {"lt": 20}},
    max_items=3,
    resolution_scale=5,
    resampling="nearest",
    num_threads=4,
)

In [ ]:
latest_stack

In [ ]:
stackstac.stack?

In [ ]:
latest_stack

In [ ]:
import numpy as np
import stackstac
from PIL import Image
from IPython.display import display
from pystac_client import Client

# Golfe du Morbihan (Brittany): west, south, east, north
bbox = (-2.95, 47.52, -2.70, 47.68)

catalog = Client.open("https://earth-search.aws.element84.com/v1")
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2024-06-01/2024-09-30",
    query={"eo:cloud_cover": {"lt": 20}},
    max_items=8,
)

items = list(search.items())
if not items:
    raise RuntimeError("No Sentinel-2 items found for this area/time window.")

# Build xarray DataArray with green + nir bands
xr_stack = stackstac.stack(
    items,
    assets=["green", "nir"],
    bounds_latlon=bbox,
    epsg=3857,
    resolution=20,
    chunksize=1024,
)

# # First scene
# scene = xr_stack.isel(time=0)

# green = scene.sel(band="green").data.compute()
# nir = scene.sel(band="nir").data.compute()

# # NDWI
# ndwi = (green - nir) / (green + nir + 1e-6)
# ndwi = np.ma.masked_invalid(ndwi)

# # Scale NDWI [-1, 1] -> [0, 255] for display
# ndwi_01 = np.ma.clip((ndwi + 1.0) / 2.0, 0.0, 1.0)
# ndwi_u8 = (ndwi_01.filled(0) * 255).astype(np.uint8)

# # Also display green band for reference (robust scaling)
# g = np.ma.masked_invalid(green)
# g_min, g_max = np.nanpercentile(g.compressed(), [2, 98]) if np.ma.count(g) else (0, 1)
# green_u8 = np.clip((g - g_min) / (g_max - g_min + 1e-6), 0, 1)
# green_u8 = (green_u8.filled(0) * 255).astype(np.uint8)

# print("Green band")
# display(Image.fromarray(green_u8, mode="L"))

# print("NDWI")
# display(Image.fromarray(ndwi_u8, mode="L"))

xr_stack